# Lab C — PR, Deployment, Public Test

**Day 2 · Estimated time: 1 hour**

Localhost proves the application works on one laptop. Deployment proves the two services can communicate over the public internet.

We will deploy from the student's own fork:

- FastAPI backend → Railway, Root Directory `smartlearn-backend`;
- React/Vite frontend → Vercel, Root Directory `smartlearn-frontend`;
- feature branch → Pull Request → the student's own fork/main.

Platform screens change over time. Preserve these source/root boundaries and use observable verification rather than guessing from dashboard labels.


> **Vibe Coding rule**
>
> For every tool-assisted task, students must first explain **inputs**, **expected output**, **allowed files/modification boundary**, and **non-goals**. Only then may Claude Code inspect the current implementation and generate the **smallest possible diff**. Whole-file rewrites are not the default.
>
> Students review every hunk with `git diff` and run the real evidence themselves: commands, `/docs`, browser Network, builds, logs, or platform checks. Claude Code saying “done” is never evidence.
>
> Full prompts are hidden references. Open one only after writing your own contract; use it to compare missing constraints, not as the first step.


## 3.1 Deployment Connection

### Purpose

Before opening Railway or Vercel, distinguish the page users open from the API that page calls. Students do not need to draw a diagram or create a deployment worksheet; they only need to explain the request path and where each variable belongs.

```text
User opens the Vercel frontend URL
        ↓  frontend reads VITE_API_URL
Vercel frontend calls the Railway backend URL
        ↓  backend checks ALLOWED_ORIGINS
Railway backend calls OpenRouter with OPENROUTER_API_KEY
```

| Configuration | Platform | Value it should contain |
|---|---|---|
| `VITE_API_URL` | Vercel | Railway backend URL |
| `ALLOWED_ORIGINS` | Railway | Vercel frontend origin |
| `OPENROUTER_API_KEY` | Railway | Secret key; never frontend code or course notes |

The backend URL is created first, so it can be placed in Vercel. The frontend URL is created second, so it can then be added to Railway's CORS allowlist. The following steps create the missing runtime definition, deploy both services, and record the real URLs.


## 3.2 Create the Railway Dockerfile

### Purpose

The starter repository does not contain `smartlearn-backend/Dockerfile`. Create this one before opening Railway. Students must first explain what the image receives, what command it must run, and that only this Dockerfile may change; then Claude Code may generate the smallest possible diff.


<!-- BEGINNER-WALKTHROUGH -->

### Step 1: Prove the gap and inspect the backend contract

From the repository root:

```bash
test ! -e smartlearn-backend/Dockerfile && echo "Dockerfile is missing as expected"
sed -n '1,220p' smartlearn-backend/main.py
sed -n '1,220p' smartlearn-backend/requirements.txt
```

Before using Claude Code, explain in your own words:

1. **Inputs:** `main.py`, `requirements.txt`, backend source, and Railway's runtime `PORT`.
2. **Expected output:** one image that installs the backend requirements and starts `main:app` on `0.0.0.0:$PORT`.
3. **Modification boundary:** create only `smartlearn-backend/Dockerfile`.
4. **Non-goals:** no repository-root requirements pointer or start command, Procfile, `railway.json`, database, authentication, or framework change.

### Step 2: Request the smallest possible diff

<details>
<summary><strong>Hidden reference prompt — create the missing Railway Dockerfile</strong> (open only after writing your own contract)</summary>

```text
Inspect smartlearn-backend/main.py and smartlearn-backend/requirements.txt first.
Do not edit yet. Explain the Docker build inputs, runtime output, and exact modification boundary.

Then create the smallest possible diff: add only smartlearn-backend/Dockerfile.
It must:
- use python:3.11-slim;
- set WORKDIR /app;
- copy and install smartlearn-backend/requirements.txt;
- copy the backend source;
- start uvicorn main:app on 0.0.0.0;
- read Railway's PORT and default to 8000 locally.

Do not add or edit repository-root deployment files, a Procfile, railway.json,
database, authentication, frontend files, or application code.
After the diff, explain each Dockerfile instruction and list the commands the student must run as evidence.
```

</details>

### Step 3: Review the created file

The minimal reference shape is:

```dockerfile
FROM python:3.11-slim

WORKDIR /app

COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

COPY . .

CMD ["sh", "-c", "uvicorn main:app --host 0.0.0.0 --port ${PORT:-8000}"]
```

Review the actual change yourself:

```bash
git diff -- smartlearn-backend/Dockerfile
sed -n '1,220p' smartlearn-backend/Dockerfile
```

Students must be able to explain the base image, working directory, dependency-cache boundary, source copy, import target, host binding, and `PORT` fallback.

**Completion evidence:** `smartlearn-backend/Dockerfile` is the only new deployment file, matches the backend's real import/dependency contract, and contains no secret or repository-root bridge.


> ✅ **Checkpoint 1 — The Railway runtime definition is ready:** `smartlearn-backend/Dockerfile` exists, matches the backend import/dependency contract, uses Railway's `PORT`, and is the only new deployment file.
>
> **If it does not match:** Fix only the Dockerfile boundary before reviewing and publishing the feature branch.


## 3.3 Review, Commit, and Push the Feature Branch

### Purpose

The cloud must receive the same source that passed locally. Students perform Git review directly; Claude Code does not decide what is safe to publish.


<!-- BEGINNER-WALKTHROUGH -->

### Step 1: Inspect every changed path

```bash
git status
git diff --stat
git diff
```

Stop if `.env`, API keys, uploaded PDFs, `uploads/`, `venv/`, `__pycache__/`, `node_modules/`, or `dist/` appear.

### Step 2: Stage only the Day 2 services

```bash
git add smartlearn-backend smartlearn-frontend
git status
git diff --staged
```

The staged diff should contain the two service directories and their real deployment files—not a repository-root dependency pointer or generated/private data.

### Step 3: Commit and push to the student's fork

```bash
git commit -m "chore: prepare Day 2 app for deployment"
git push -u origin feature/day2-lite
```

Open the branch in the student's own fork and verify the latest commit is visible.


> ✅ **Checkpoint 2 — The feature branch is secure and reproducible:** GitHub contains the locally tested Day 2 source and excludes secrets, uploaded files, environments, dependencies, and build output.
>
> **Security stop:** If an API key was committed, revoke it first and ask an instructor for history cleanup. A later deletion commit does not remove the exposed secret from history.


## 3.4 Open, Review, and Merge the First Pull Request

### Purpose

Merge the reviewed feature into the student's own fork/main before connecting production platforms. Do not target the teacher/upstream repository.


<!-- BEGINNER-WALKTHROUGH -->

### Step 1: Open the Pull Request on the student's fork

Verify the GitHub repository selector before creating the PR:

- **base repository:** the student's own fork;
- **base branch:** `main`;
- **compare branch:** `feature/day2-lite`.

The feature branch proposes changes into the student's fork/main. It must not target teacher/upstream main.

### Step 2: Draft the description from evidence

Ask Claude Code to draft What / How / Proof / Limit from the reviewed diff and these human-run results: backend `/health`, local upload/chat with citations, and frontend build. Students verify every claim and remove secrets/private PDF text.

### Step 3: Human review before merge

Confirm both service directories, Dockerfile, request contracts, temporary-state limitation, and secret exclusions. Reject any root dependency pointer or unrelated infrastructure.

### Step 4: Merge

Merge into the student's own fork/main and record the PR link.


> ✅ **Checkpoint 3 — Day 2 is merged into the student's fork/main:** The PR has a verified What / How / Proof / Limit description, reviewed diff, no secrets, and a completed merge to the student's repository.
>
> **If it does not match:** Correct the base repository/branch instead of merging into teacher/upstream main.


## 3.5 Synchronize Local `main`

### Purpose

The merge happened in the student's fork. Update the local repository so it matches the student's fork/main that Railway and Vercel will deploy.


In [ ]:
git switch main
git pull
git log -3 --oneline
git status


> **Expected output:** Recent history includes the merged Day 2 work and `git status` is clean.
>
> From this point forward, connect both platforms to the student's own fork/main. If a dashboard shows teacher/upstream, another fork, or a feature branch, correct the source before trusting the deployment.


## 3.6 Deploy the Backend to Railway

### Purpose

Create a public FastAPI service from the merged `main` branch. Students operate Railway directly. Claude Code is used only after a real log or test provides evidence of a code problem.


<!-- BEGINNER-WALKTHROUGH -->

### Step 1: Create the Railway service from the student's fork

1. Create a Railway account (https://railway.com/) and link it to your GitHub.
2. Choose **Deploy from GitHub Repo** and select the student's SmartLearn fork.
3. Confirm the production branch is `main`.
4. In service settings, set **Root Directory** to: `smartlearn-backend`

Railway must build from this directory and read `smartlearn-backend/Dockerfile`. Do not deploy from the repository root and do not add a repository-root start command.

### Step 2: Add backend variables

| Name | Initial value | Secret? |
|---|---|---|
| `OPENROUTER_API_KEY` | student's real OpenRouter key | yes |
| `ALLOWED_ORIGINS` | `http://localhost:5173` initially | no |

Do not create `PORT`; Railway supplies it. Never paste the key into Claude Code, GitHub, screenshots, or the notebook.

### Step 3: Deploy through the Dockerfile

Apply the source/root settings and deploy. Read the build log to confirm Railway found the Dockerfile in the configured Root Directory, installed backend dependencies, and started the service. The Dockerfile—not a dashboard command copied from repository-root instructions—is the runtime definition.

### Step 4: Generate the public backend URL

Under Public Networking, generate and copy the complete Railway domain. Call it `BACKEND_URL`.


### Verify Railway before touching Vercel

Run this public sequence in Railway `/docs`:

1. open `BACKEND_URL/health` and confirm `200 {"ok": true}`;
2. upload a known text PDF using `/upload?chat_id=day2-demo`;
3. call `/chat` with `message + chat_id`;
4. confirm `answer + citations` includes valid page evidence;
5. ask an unknown question and confirm the response does not invent evidence.

This proves Dockerfile startup, dependencies, environment key, PDF handling, the shared chat contract, citations, and unknown-information behavior independently of Vercel.

### Evidence-driven failure handling

- Dockerfile/build failed → inspect the first file or dependency error.
- Service crashed → inspect the Dockerfile start/port log.
- `/health` works but `/chat` fails → inspect the runtime key/upstream error.
- Upload succeeds but chat cannot find state → confirm the same `chat_id` and whether Railway restarted.

Only after collecting the exact phase and relevant log should students ask Claude Code to diagnose.


> ✅ **Checkpoint 4 — Railway works independently:** Root Directory is `smartlearn-backend`, its Dockerfile starts the service, and the public `/health → upload → /chat → citations → unknown question` sequence passes.
>
> **Do not continue to Vercel** until this checkpoint passes.


## 3.7 Understand the Hosting Limitation

### Purpose

Deployment makes the prototype reachable; it does not make temporary state persistent.

The backend uses local/in-memory chat state. Railway may restart after a deploy or environment-variable update, which clears uploaded content. For Day 2 this is expected: re-upload the PDF with the classroom `chat_id` before calling `/chat` again.

Do not add PostgreSQL, object storage, Redis, or authentication during this checkpoint.


> ✅ **Checkpoint 5 — The limitation is explained honestly:** Students can explain why an old `chat_id` may return 404 after a restart and why re-uploading—not pretending persistence—is the Day 2 response.


## 3.8 Deploy the Frontend to Vercel

### Purpose

Build the React/Vite frontend with the verified Railway domain. The platform configuration is performed directly; Claude Code is reserved for explaining a real build error.


<!-- BEGINNER-WALKTHROUGH -->

### Step 1: Import the student's fork

1. Create a Vercel account (https://vercel.com/) and link it to your GitHub.
2. Choose **Add New → Project** in Vercel.
3. Import the student's SmartLearn fork.
4. Confirm the production branch is `main`.

### Step 2: Configure the frontend Root Directory

| Setting | Workshop value |
|---|---|
| Root Directory | `smartlearn-frontend` |
| Framework Preset | Vite (normally detected) |
| Build Command | `npm run build` |
| Output Directory | `dist` |

Do not point Vercel at `smartlearn-backend` or the repository root.

### Step 3: Add the frontend environment variable

```text
Name:  VITE_API_URL
Value: the complete Railway BACKEND_URL
```

Include `https://`. Do not append `/docs` or `/health`, and never place `OPENROUTER_API_KEY` in Vercel.

### Step 4: Deploy and copy the frontend URL

Deploy, open the generated Vercel address, and copy its scheme + hostname as `FRONTEND_URL`.


### Step 5: Inspect the first production request

Open the Vercel page, then open browser DevTools → **Network** and try an upload.

First confirm the request URL begins with the Railway `BACKEND_URL`. If it points to localhost or Vercel itself, correct `VITE_API_URL` and redeploy. Vercel environment-variable changes apply only to a new deployment.

At this moment the request may be blocked by CORS because Railway still allows only localhost. That is expected: the frontend URL did not exist until Vercel generated it.


> ✅ **Checkpoint 6 — Vercel targets the correct backend:** Root Directory is `smartlearn-frontend`, the public page loads, and Network sends API requests to the Railway domain even if CORS remains the expected blocker.
>
> **If it does not match:** Fix the Vercel Root Directory or `VITE_API_URL`, redeploy, and inspect Network before changing React code.


## 3.9 Close the CORS Loop

### Purpose

Now the real frontend origin exists. Add it to the backend allowlist without weakening the policy to `*`.


<!-- BEGINNER-WALKTHROUGH -->

### Step 1: Identify the exact Vercel origin

Copy scheme + hostname only, for example `https://smartlearn-lite-yourname.vercel.app`.

### Step 2: Update Railway CORS

Set `ALLOWED_ORIGINS` to the exact production origin (and localhost if desired):

```text
http://localhost:5173, https://smartlearn-lite-yourname.vercel.app
```

Applying the variable change restarts/redeploys Railway. Because the backend keeps chat state in memory, the earlier upload is cleared. This is expected—not a bug.

### Step 3: Re-upload, then re-test

After Railway is healthy again:

1. re-upload the PDF with `/upload?chat_id=day2-demo`;
2. ask a known question through `/chat` and verify citations;
3. ask an unknown question and verify the response does not invent evidence.

Inspect Network. `/upload?chat_id=` and `/chat` should return application responses rather than browser CORS blocks.

### If CORS still fails

Collect page origin, request URL/method, preflight status, current non-secret allowlist, and middleware parsing. Reject wildcard CORS as a shortcut.


> ✅ **Checkpoint 7 — The public loop is closed:** The Vercel origin is allowed, the expected Railway restart is complete, the PDF was re-uploaded, and public `/chat` returns cited known answers plus honest unknown answers.
>
> **If it does not match:** Compare browser origin, `ALLOWED_ORIGINS`, and `VITE_API_URL`; then confirm restart and re-upload occurred.


## 3.10 Run the Production Acceptance Test

### Purpose

Turn “it seems live” into repeatable evidence. Use the public Vercel page unless a row explicitly names Railway `/docs`.


<!-- BEGINNER-WALKTHROUGH -->

Run and record this public acceptance sequence without secrets/private document text:

| Order | Action | Expected evidence |
|---:|---|---|
| 1 | open Railway `/health` | status 200 and `{"ok": true}` |
| 2 | upload a known text PDF under 30 pages | `/upload?chat_id=day2-demo` returns success |
| 3 | send a known question to `/chat` | answer is grounded in the uploaded PDF |
| 4 | inspect citations | cited page numbers are valid |
| 5 | send an unknown question | response admits insufficient evidence |
| 6 | repeat through public Vercel UI | Network targets Railway and UI shows answer/citations |

Also record the student's fork/main commit, PR link, Railway/Vercel URLs, Root Directories, and the expected restart/re-upload behavior after CORS changes.


> ✅ **Checkpoint 8 — Day 2 is merged and live:** The student's fork/main feeds both platforms; Railway uses `smartlearn-backend`, Vercel uses `smartlearn-frontend`, and the public `/health → upload → /chat → citations → unknown question` sequence passes after any required re-upload.
>
> **If a later deployment breaks:** Verify `/health`, re-upload after restart, then diagnose one layer at a time.


## 3.11 Record the Evidence and Prepare the Demo

Complete the non-secret verification record in Appendix D and rehearse the demo in Appendix G. Record the actual URLs, Root Directories, student's fork/main commit, and PR—not placeholders.

Before leaving, confirm both platforms still track the student's fork/main. Feature branches remain review work rather than silently replacing production.


## 3.12 Official Documentation Reference

Platform labels and defaults change. Preserve the workshop architecture and use current official documentation:

- Railway Dockerfiles: https://docs.railway.com/guides/dockerfiles
- Railway services and GitHub source: https://docs.railway.com/services
- Railway monorepo/root directories: https://docs.railway.com/guides/monorepo
- Vercel Git deployment: https://vercel.com/docs/git
- Vercel Vite guide: https://vercel.com/docs/frameworks/vite
- Vercel environment variables: https://vercel.com/docs/environment-variables
- FastAPI CORS: https://fastapi.tiangolo.com/tutorial/cors/

Official docs are authoritative for current control names; this notebook is authoritative for the workshop's two Root Directories and test sequence.


<!-- BEGINNER-WALKTHROUGH -->

### Use official docs without losing the workshop path

When a platform button differs:

1. Name the concept you need: repository source, branch, start command, environment variable, root directory, public domain, or deployment URL.
2. Open the matching official link above.
3. Search for that concept.
4. Apply only the smallest platform setting required by this repository.
5. Return to the next observable checkpoint: build log, `/health`, Network request, or public UI.

Do not copy a large enterprise deployment template into the beginner project. Our Day 2 deployment intentionally uses the smallest configuration that proves the vertical slice.


# Homework

Use the same selectable-text PDF that passed the public deployment test.

| # | Test question | Expected page |
|---:|---|---|
| 1 | What is the dimension of each attention head used in the Transformer? | Page 5 |
| 2 | What optimizer and learning-rate schedule are used to train the Transformer? | Page 7 |
| 3 | What regularization methods are used to train the Transformer? | Page 7 |
| 4 | What accuracy does the Transformer achieve on the ImageNet image-classification benchmark? | None — the answer must admit insufficient evidence |

# Appendix A: Build Time versus Runtime Variables

The frontend and backend read configuration at different moments.

## Frontend

Vite reads `VITE_API_URL` during the build. The resulting browser bundle contains that public URL. If the Vercel value changes, create a new deployment so a new bundle is built.

## Backend

FastAPI reads `OPENROUTER_API_KEY` and `ALLOWED_ORIGINS` while the Railway service runs. A variable change may restart or redeploy the service.

### Prediction questions

1. `VITE_API_URL` changes but Vercel is not redeployed. Which URL does the existing bundle use?
2. An API key is stored in a variable beginning with `VITE_`. Can a browser user inspect it?
3. `ALLOWED_ORIGINS` changes but the backend has not restarted. What symptom may remain?
4. An allowed origin includes `/app`. Why does it not match a browser origin?


# Appendix B: Deployment Log Reading

Hosting logs contain many routine lines and usually one useful failure. Start with the first error that refers to the application.

## Build phase

- dependency file not found;
- package installation failure;
- frontend import/build error;
- wrong Vercel root directory.

## Start phase

- module or `app` object cannot be imported;
- Uvicorn start command is wrong;
- server binds to the wrong host or port;
- process exits immediately.

## Request phase

- missing runtime variable;
- PDF parser exception;
- upstream model timeout/error;
- route returns an unexpected status.

Copy only the relevant error and a few surrounding lines. Remove keys and private PDF text, state the exact phase and deployed commit, and change one hypothesis at a time.


# Appendix C: Full-Stack Configuration Triangle

A public browser request succeeds only if three real values agree:

```text
Browser page origin = FRONTEND_URL
Railway ALLOWED_ORIGINS contains FRONTEND_URL
Vercel VITE_API_URL = BACKEND_URL
```

Distinctive mismatches:

1. `VITE_API_URL` points to localhost → each visitor's browser calls its own computer.
2. `ALLOWED_ORIGINS` omits the Vercel origin → Railway `/docs` works but the browser blocks the cross-origin request.
3. `VITE_API_URL` points to Vercel → the static frontend host receives `/upload`, which belongs to FastAPI.

Use Network evidence to identify the broken side of the triangle; do not rewrite both applications.


# Appendix D: Production Verification Record

Create `docs/day2_deployment.md` without secrets:

```markdown
# Day 2 Deployment

## URLs
- Frontend: ...
- Backend health: .../health
- Backend docs: .../docs

## Source
- Repository: student's fork
- Deployed branch / merge target: main
- Merged commit: ...
- Pull Request: ...

## Root Directories
- Railway: smartlearn-backend
- Vercel: smartlearn-frontend

## Environment variable names
- Railway: OPENROUTER_API_KEY, ALLOWED_ORIGINS
- Vercel: VITE_API_URL

## Acceptance results
- /health: pass/fail
- Upload: pass/fail
- Known /chat + citations: pass/fail + expected page
- Unknown question: pass/fail
- CORS restart + re-upload recovery: pass/fail

## Known limitations
- Railway restart clears in-memory uploaded/chat state; re-upload is expected.
```


# Appendix E: Rollback and Recovery Thinking

A deployment can succeed technically and still break the product. Before deploying, identify the last known-good commit.

### Scenario 1 — frontend broke, backend healthy

Evidence: backend `/health` and `/docs` work; Vercel build or UI fails. Recovery focuses on frontend commit/configuration.

### Scenario 2 — backend failed to start

Evidence: health URL fails; Railway logs show import or start error. Recovery focuses on start command, dependencies, or backend commit.

### Scenario 3 — both deploy, integration fails

Evidence: pages load independently; browser Network shows CORS or wrong URL. Recovery focuses on configuration triangle.

### Safe workshop response

Do not rewrite both services under time pressure. Restore or redeploy the last known-good source/configuration, verify the milestone, then diagnose the broken change separately. Rollback is not failure; it is a controlled engineering tool.


# Appendix F: Pull Request Review Rubric

Score each dimension 0–2:

| Dimension | 0 | 1 | 2 |
|---|---|---|---|
| Scope | unrelated features included | mostly Day 2 | exact vertical slice |
| Evidence | claims only | vague test claim | reproducible local test evidence |
| Security | secret/upload exposed | unclear ignore state | explicit checks passed |
| Architecture | responsibilities mixed | partial separation | route/PDF/LLM/UI boundaries clear |
| Failure handling | crashes or vague errors | some errors | planned input/lifecycle/upstream errors |
| Learning clarity | author cannot explain | explains main flow | explains choices and limitations |

A PR is ready when the author can walk through the local user flow, show reproducible tests, explain limitations, and identify where each failure would be observed. Public acceptance evidence is recorded after the reviewed PR is merged and deployed.


# Appendix G: Day 2 Demo Script

Keep the demo under three minutes:

1. Show Railway `/health`.
2. Upload a known PDF.
3. Send a known question through `/chat` and point to citations.
4. Send an unknown question and show the honest response.
5. Explain Vercel → Railway → OpenRouter and the two Root Directories.
6. State that Railway restart clears memory and requires re-upload.

Do not spend demo time scrolling through code or expose secrets/private PDFs.


# Appendix H: Deployment Teach-Back Questions

1. Why deploy the backend before the frontend?
2. Why can `/docs` work while the React page is blocked?
3. Which environment variable is public by design?
4. Which variable must never enter the browser build?
5. Why might an uploaded document disappear in Railway?
6. What does `$PORT` represent?
7. Which branch is production currently tracking?
8. What evidence proves the deployed app—not localhost—called OpenRouter?
9. How would you distinguish a Vercel build error from a runtime API error?
10. Which known limitation gives Day 3 its main technical problem?

Students should answer with an observable check or URL whenever possible.


## Final Checkpoint

- [ ] The reviewed PR is merged into the student's own fork/main; Railway uses Root Directory `smartlearn-backend` and its Dockerfile, while Vercel uses `smartlearn-frontend`.
- [ ] The public Railway sequence passes: `/health → upload with day2-demo → /chat → valid citations → honest unknown answer`.
- [ ] Vercel targets the Railway URL; after the CORS-triggered Railway restart, the PDF is re-uploaded and the complete browser flow passes.
- [ ] The evidence record contains the merged commit, PR, URLs, Root Directories, and test results, while Git contains no secret, uploaded PDF, dependency directory, or generated output.

If these four statements are true, Day 2 is reviewed, deployed, and publicly verified.
